In [4]:
# Text vs bytes: how UTF-8 stores characters
text = "hi क"
data = text.encode("utf-8")

print("text      :", text)
print("bytes     :", data)
print("as numbers:", list(map(int, data)))
print("characters:", len(text))
print("bytes     :", len(data))
for b in data:
    print(f"  {b:3d}  ->  {b:08b}")

text      : hi क
bytes     : b'hi \xe0\xa4\x95'
as numbers: [104, 105, 32, 224, 164, 149]
characters: 4
bytes     : 6
  104  ->  01101000
  105  ->  01101001
   32  ->  00100000
  224  ->  11100000
  164  ->  10100100
  149  ->  10010101


In [5]:
# Bytes per character in different scripts
for ch in ["a", "Z", " ", "क", "ा", "😄"]:
    encoded = ch.encode("utf-8")
    print(repr(ch), "->", list(encoded), f"({len(encoded)} bytes)")

'a' -> [97] (1 bytes)
'Z' -> [90] (1 bytes)
' ' -> [32] (1 bytes)
'क' -> [224, 164, 149] (3 bytes)
'ा' -> [224, 164, 190] (3 bytes)
'😄' -> [240, 159, 152, 132] (4 bytes)


In [6]:
# Adjacent pairs: the clear way and the zip way
ids = [104, 105, 32, 104, 105]

# Way 1: simple loop
pairs_loop = []
for i in range(len(ids) - 1):
    pairs_loop.append((ids[i], ids[i + 1]))

# Way 2: zip (Karpathy style)
pairs_zip = list(zip(ids, ids[1:]))

print("ids       :", ids)
print("ids[1:]   :", ids[1:])
print("loop pairs:", pairs_loop)
print("zip pairs :", pairs_zip)
print("same?     :", pairs_loop == pairs_zip)

ids       : [104, 105, 32, 104, 105]
ids[1:]   : [105, 32, 104, 105]
loop pairs: [(104, 105), (105, 32), (32, 104), (104, 105)]
zip pairs : [(104, 105), (105, 32), (32, 104), (104, 105)]
same?     : True


In [7]:
# Part 1: numbers -> bytes -> text
nums = [104, 105, 32, 224, 164, 149]

as_bytes = bytes(nums)
print("bytes(nums):", as_bytes)
print("decoded    :", as_bytes.decode("utf-8"))

# Part 2: joining byte pieces together
pieces = [b"hi", b" ", bytes([224, 164]), bytes([149])]
joined = b"".join(pieces)
print("\npieces     :", pieces)
print("joined     :", joined)
print("decoded    :", joined.decode("utf-8"))

# Part 3: decoding half a character
broken = bytes([224, 164])
print("\nbroken     :", broken.decode("utf-8", errors="replace"))

bytes(nums): b'hi \xe0\xa4\x95'
decoded    : hi क

pieces     : [b'hi', b' ', b'\xe0\xa4', b'\x95']
joined     : b'hi \xe0\xa4\x95'
decoded    : hi क

broken     : �


In [22]:
# Training text: same words as the hand-worked example
train_text = (
    "low low low low low "
    "lower lower "
    "newest newest newest newest newest newest "
    "widest widest widest"
)

# Text -> UTF-8 bytes -> list of ids (0-255)
ids = list(train_text.encode("utf-8"))
print("Total ids:", len(ids))
print("First 20 :", ids[:20])


def get_stats(ids):
    """Count how often each adjacent pair of ids appears."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        if pair in counts:
            counts[pair] += 1
        else:
            counts[pair] = 1
    print("\nPair counts:", counts)
    return counts


stats = get_stats(ids)

# Find the most frequent pair (simple loop way)
best_pair = None
best_count = 0
for pair, count in stats.items():
    if count > best_count:
        best_pair = pair
        best_count = count

print(
    "\nMost frequent pair:", best_pair, "->", bytes(best_pair), f"({best_count} times)"
)

Total ids: 94
First 20 : [108, 111, 119, 32, 108, 111, 119, 32, 108, 111, 119, 32, 108, 111, 119, 32, 108, 111, 119, 32]

Pair counts: {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 8, (101, 114): 2, (114, 32): 2, (32, 110): 6, (110, 101): 6, (101, 119): 6, (101, 115): 9, (115, 116): 9, (116, 32): 8, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 101): 3}

Most frequent pair: (101, 115) -> b'es' (9 times)


In [9]:
def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`."""
    new_ids = []
    i = 0
    while i < len(ids):
        # Is there a matching pair starting at position i?
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_id)
            i += 2  # skip both items of the pair
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# Small test first
print("Small test:", merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

# Now on our training ids
merged_ids = merge(ids, best_pair, 256)
print("\nBefore      :", len(ids), "ids")
print("After       :", len(merged_ids), "ids")
print("How many 256:", merged_ids.count(256))

Small test: [5, 6, 99, 9, 1]

Before      : 94 ids
After       : 85 ids
How many 256: 9


In [26]:
def train_bpe(text, vocab_size, verbose=True):
    """Learn BPE merges on raw UTF-8 bytes until the vocab reaches vocab_size."""
    num_merges = vocab_size - 256
    ids = list(text.encode("utf-8"))
    # print(ids)

    # merges: (id1, id2) -> new_id, kept in the order they were learned
    merges = {}

    # vocab: id -> bytes. Start with the 256 single bytes.
    vocab = {}
    for i in range(256):
        vocab[i] = bytes([i])
    #     print(i, vocab[i], vocab[i].decode("utf-8", errors="replace"))

    # print(vocab)
    for step in range(num_merges):
        stats = get_stats(ids)
        if not stats:
            break  # nothing left to merge

        best_pair = max(stats, key=stats.get)
        new_id = 256 + step

        ids = merge(ids, best_pair, new_id)
        merges[best_pair] = new_id
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]

        if verbose:
            count = stats[best_pair]
            print(
                f"Merge {step + 1}: {best_pair} -> {new_id}  {vocab[new_id]!r}  ({count} times)"
            )

    return merges, vocab, ids


merges, vocab, trained_ids = train_bpe(train_text, vocab_size=266)

print("\nLength before:", len(train_text.encode("utf-8")))
print("Length after :", len(trained_ids))
print("Vocab size   :", len(vocab))


Pair counts: {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 8, (101, 114): 2, (114, 32): 2, (32, 110): 6, (110, 101): 6, (101, 119): 6, (101, 115): 9, (115, 116): 9, (116, 32): 8, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 101): 3}
Merge 1: (101, 115) -> 256  b'es'  (9 times)

Pair counts: {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2, (101, 114): 2, (114, 32): 2, (32, 110): 6, (110, 101): 6, (101, 119): 6, (119, 256): 6, (256, 116): 9, (116, 32): 8, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 256): 3}
Merge 2: (256, 116) -> 257  b'est'  (9 times)

Pair counts: {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2, (101, 114): 2, (114, 32): 2, (32, 110): 6, (110, 101): 6, (101, 119): 6, (119, 257): 6, (257, 32): 8, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 257): 3}
Merge 3: (257, 32) -> 258  b'est '  (8 times)

Pair counts: {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2

In [27]:
import re

# Simple split pattern:
#   1. " ?[^\s.,!?]+"  -> a word (anything that is not space or punctuation), with an optional space in front
#   2. " ?[.,!?]+"     -> punctuation, with an optional space in front
#   3. "\s+"           -> any leftover spaces or new lines
SPLIT_PATTERN = r" ?[^\s.,!?]+| ?[.,!?]+|\s+"

samples = [
    "low low lower newest widest",
    "hi there! how are you?",
    "मैं ठीक हूँ, तुम कैसे हो?",
]

for s in samples:
    print(re.findall(SPLIT_PATTERN, s))

['low', ' low', ' lower', ' newest', ' widest']
['hi', ' there', '!', ' how', ' are', ' you', '?']
['मैं', ' ठीक', ' हूँ', ',', ' तुम', ' कैसे', ' हो', '?']


In [30]:
# ============================================================
# Final mini BPE tokenizer: byte-level BPE with regex chunks
# ============================================================
import re

# Split text into words / punctuation / spaces before BPE,
# so merges never cross word boundaries.
SPLIT_PATTERN = r" ?[^\s.,!?]+| ?[.,!?]+|\s+"


def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`."""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_id)
            i += 2  # skip both items of the pair
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


def train_bpe(text, vocab_size, verbose=True):
    """Learn BPE merges on UTF-8 bytes, never merging across regex chunks."""
    num_merges = vocab_size - 256

    # 1. Split text into chunks, then turn each chunk into a list of byte ids
    chunks = re.findall(SPLIT_PATTERN, text)
    chunk_ids = []
    for chunk in chunks:
        chunk_ids.append(list(chunk.encode("utf-8")))

    # merges: (id1, id2) -> new_id, kept in the order they were learned
    merges = {}

    # vocab: id -> bytes. Start with the 256 single bytes.
    vocab = {}
    for i in range(256):
        vocab[i] = bytes([i])

    for step in range(num_merges):
        # 2. Count pairs INSIDE each chunk only
        stats = {}
        for ids in chunk_ids:
            for pair in zip(ids, ids[1:]):
                stats[pair] = stats.get(pair, 0) + 1
        if not stats:
            break  # nothing left to merge

        best_pair = max(stats, key=stats.get)
        new_id = 256 + step

        # 3. Merge inside each chunk
        new_chunk_ids = []
        for ids in chunk_ids:
            new_chunk_ids.append(merge(ids, best_pair, new_id))
        chunk_ids = new_chunk_ids

        # 4. Save the merge rule and the new token's bytes
        merges[best_pair] = new_id
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]

        if verbose:
            count = stats[best_pair]
            print(
                f"Merge {step + 1}: {best_pair} -> {new_id}  {vocab[new_id]!r}  ({count} times)"
            )

    # 5. Join all chunks back into one list of ids
    all_ids = []
    for ids in chunk_ids:
        all_ids.extend(ids)

    return merges, vocab, all_ids


def decode(ids, vocab):
    """Turn a list of token ids back into text."""
    pieces = []
    for token_id in ids:
        pieces.append(vocab[token_id])  # id -> bytes
    raw = b"".join(pieces)  # glue all bytes together first
    return raw.decode("utf-8", errors="replace")  # then decode once


# ------------------------------------------------------------
# Train the final tokenizer
# ------------------------------------------------------------
train_text = (
    "low low low low low "
    "lower lower "
    "newest newest newest newest newest newest "
    "widest widest widest"
)

print("Chunks:", re.findall(SPLIT_PATTERN, train_text))
print()

merges, vocab, train_ids = train_bpe(train_text, vocab_size=266)

print("\nLength before :", len(train_text.encode("utf-8")))
print("Length after  :", len(train_ids))
print("Vocab size    :", len(vocab))

text_back = decode(train_ids, vocab)
print("\nDecoded        :", text_back)
print("Same as input? :", text_back == train_text)

Chunks: ['low', ' low', ' low', ' low', ' low', ' lower', ' lower', ' newest', ' newest', ' newest', ' newest', ' newest', ' newest', ' widest', ' widest', ' widest']

Merge 1: (101, 115) -> 256  b'es'  (9 times)
Merge 2: (256, 116) -> 257  b'est'  (9 times)
Merge 3: (108, 111) -> 258  b'lo'  (7 times)
Merge 4: (258, 119) -> 259  b'low'  (7 times)
Merge 5: (32, 259) -> 260  b' low'  (6 times)
Merge 6: (32, 110) -> 261  b' n'  (6 times)
Merge 7: (261, 101) -> 262  b' ne'  (6 times)
Merge 8: (262, 119) -> 263  b' new'  (6 times)
Merge 9: (263, 257) -> 264  b' newest'  (6 times)
Merge 10: (32, 119) -> 265  b' w'  (3 times)

Length before : 94
Length after  : 29
Vocab size    : 266

Decoded        : low low low low low lower lower newest newest newest newest newest newest widest widest widest
Same as input? : True


In [35]:
import re


def encode_a(text, merges, verbose=False):
    """Way A: split text into words, then apply each merge rule to every word."""
    # 1. Split text into words using the regex, then turn each word into byte ids
    words = []
    for chunk in re.findall(SPLIT_PATTERN, text):
        words.append(list(chunk.encode("utf-8")))

    # 2. Outer loop: take merge rules one by one (in learning order)
    for pair, new_id in merges.items():
        # 3. Inner loop: apply this rule to every word
        for w in range(len(words)):
            word = words[w]
            new_word = []
            i = 0
            while i < len(word):
                # Pair found here? Replace both items with new_id
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(new_id)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1

            if verbose and new_word != word:
                print(f"Rule {pair} -> {new_id}  |  {word}  ->  {new_word}")

            words[w] = new_word  # save the updated word back

    # 4. Join all words into one flat list of ids
    all_ids = []
    for word in words:
        all_ids.extend(word)
    return all_ids


def encode_b(text, merges):
    """Way B (Karpathy): keep merging the present pair that was learned earliest."""
    all_ids = []
    for chunk in re.findall(SPLIT_PATTERN, text):
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            # Among the pairs in ids, find the one with the smallest merge id
            best_pair = None
            best_rank = None
            for pair in zip(ids, ids[1:]):
                if pair in merges:
                    rank = merges[pair]
                    if best_rank is None or rank < best_rank:
                        best_pair = pair
                        best_rank = rank
            if best_pair is None:
                break  # no merge rule fits any more
            ids = merge(ids, best_pair, best_rank)
        all_ids.extend(ids)
    return all_ids


def show(ids, vocab):
    """Show each token id as its bytes, so we can read the tokens."""
    pieces = []
    for token_id in ids:
        pieces.append(vocab[token_id])
    return pieces


# Watch Way A step by step on one example
print("Way A, step by step:")
ids = encode_a("low lowest", merges, verbose=True)
print("Result:", ids, show(ids, vocab))
print()

# Compare both ways on several texts
tests = ["lowest", "newer", "low newer", "slow", "low newest widest", "hello world!"]

for t in tests:
    ids_a = encode_a(t, merges)
    ids_b = encode_b(t, merges)
    print(f"Text    : {t!r}")
    print(f"Way A   : {ids_a}")
    print(f"Way B   : {ids_b}")
    print(f"Tokens  : {show(ids_a, vocab)}")
    print(f"A == B? : {ids_a == ids_b}   |   decode OK? : {decode(ids_a, vocab) == t}")
    print()

Way A, step by step:
Rule (101, 115) -> 256  |  [32, 108, 111, 119, 101, 115, 116]  ->  [32, 108, 111, 119, 256, 116]
Rule (256, 116) -> 257  |  [32, 108, 111, 119, 256, 116]  ->  [32, 108, 111, 119, 257]
Rule (108, 111) -> 258  |  [108, 111, 119]  ->  [258, 119]
Rule (108, 111) -> 258  |  [32, 108, 111, 119, 257]  ->  [32, 258, 119, 257]
Rule (258, 119) -> 259  |  [258, 119]  ->  [259]
Rule (258, 119) -> 259  |  [32, 258, 119, 257]  ->  [32, 259, 257]
Rule (32, 259) -> 260  |  [32, 259, 257]  ->  [260, 257]
Result: [259, 260, 257] [b'low', b' low', b'est']

Text    : 'lowest'
Way A   : [259, 257]
Way B   : [259, 257]
Tokens  : [b'low', b'est']
A == B? : True   |   decode OK? : True

Text    : 'newer'
Way A   : [110, 101, 119, 101, 114]
Way B   : [110, 101, 119, 101, 114]
Tokens  : [b'n', b'e', b'w', b'e', b'r']
A == B? : True   |   decode OK? : True

Text    : 'low newer'
Way A   : [259, 263, 101, 114]
Way B   : [259, 263, 101, 114]
Tokens  : [b'low', b' new', b'e', b'r']
A == B? : Tr

In [34]:
import tiktoken
from tokenizers import Tokenizer

# OpenAI tokenizers (via tiktoken)
gpt4 = tiktoken.get_encoding("cl100k_base")  # used by GPT-4
gpt4o = tiktoken.get_encoding("o200k_base")  # used by GPT-4o (newer, bigger vocab)

# A Hugging Face tokenizer (downloads tokenizer.json the first time)
qwen = Tokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

print("GPT-4  vocab size :", gpt4.n_vocab)
print("GPT-4o vocab size :", gpt4o.n_vocab)
print("Qwen2.5 vocab size:", qwen.get_vocab_size())

sample = "I am learning"
print("\nSample:", sample)

for name, enc in [("GPT-4", gpt4), ("GPT-4o", gpt4o)]:
    ids = enc.encode(sample)
    pieces = []
    for t in ids:
        pieces.append(enc.decode_single_token_bytes(t))
    print(f"\n{name}: {len(ids)} tokens")
    print("  ids   :", ids)
    print("  pieces:", pieces)

qwen_ids = qwen.encode(sample).ids
print(f"\nQwen2.5: {len(qwen_ids)} tokens")
print("  ids   :", qwen_ids)

GPT-4  vocab size : 100277
GPT-4o vocab size : 200019
Qwen2.5 vocab size: 151665

Sample: I am learning

GPT-4: 3 tokens
  ids   : [40, 1097, 6975]
  pieces: [b'I', b' am', b' learning']

GPT-4o: 3 tokens
  ids   : [40, 939, 7524]
  pieces: [b'I', b' am', b' learning']

Qwen2.5: 3 tokens
  ids   : [40, 1079, 6832]


In [36]:
# Same idea in different "languages": how many tokens does each tokenizer need?
texts = {
    "English": "I am learning how large language models work, and today I built my own tokenizer.",
    "Hinglish": "Main seekh raha hoon ki large language models kaise kaam karte hain, aur aaj maine apna tokenizer banaya.",
    "Hindi": "मैं सीख रहा हूँ कि बड़े भाषा मॉडल कैसे काम करते हैं, और आज मैंने अपना टोकनाइज़र बनाया।",
    "Python code": "def count_tokens(text):\n    return len(tokenizer.encode(text))",
}


def count_all(text):
    """Count tokens for one text with every tokenizer."""
    return {
        "GPT-4": len(gpt4.encode(text)),
        "GPT-4o": len(gpt4o.encode(text)),
        "Qwen2.5": len(qwen.encode(text).ids),
    }


results = {}
for name, text in texts.items():
    results[name] = count_all(text)

# Print as a markdown table (ready to paste into notes/day02.md)
print("| Text | Words | Bytes | GPT-4 | GPT-4o | Qwen2.5 |")
print("|------|-------|-------|-------|--------|---------|")
for name, text in texts.items():
    words = len(text.split())
    n_bytes = len(text.encode("utf-8"))
    r = results[name]
    print(
        f"| {name} | {words} | {n_bytes} | {r['GPT-4']} | {r['GPT-4o']} | {r['Qwen2.5']} |"
    )

# How many times more tokens than English?
print("\nTokens compared to English (1.00x = same as English):")
for name in texts:
    line = f"  {name:12}"
    for tok in ["GPT-4", "GPT-4o", "Qwen2.5"]:
        ratio = results[name][tok] / results["English"][tok]
        line += f"  {tok}: {ratio:.2f}x"
    print(line)

| Text | Words | Bytes | GPT-4 | GPT-4o | Qwen2.5 |
|------|-------|-------|-------|--------|---------|
| English | 15 | 81 | 17 | 17 | 17 |
| Hinglish | 18 | 105 | 31 | 29 | 31 |
| Hindi | 18 | 222 | 92 | 24 | 86 |
| Python code | 4 | 62 | 13 | 13 | 13 |

Tokens compared to English (1.00x = same as English):
  English       GPT-4: 1.00x  GPT-4o: 1.00x  Qwen2.5: 1.00x
  Hinglish      GPT-4: 1.82x  GPT-4o: 1.71x  Qwen2.5: 1.82x
  Hindi         GPT-4: 5.41x  GPT-4o: 1.41x  Qwen2.5: 5.06x
  Python code   GPT-4: 0.76x  GPT-4o: 0.76x  Qwen2.5: 0.76x
